# All 8 extraction algorithms

Same τ and τ_log build as `raw_vs_log.ipynb`, plus four **aggregate** methods that use the entire (i, v) slice of τ instead of picking one witness coordinate.

**Single-witness methods** (use one $(i_j, v_j)$ per $j$):
1. max-spread + raw cost
2. max-min spread + raw cost  (top1 − top2 margin)
3. max-spread + log cost
4. max-min spread + log cost

**Aggregate methods** (use all $n^2$ cells in the (i, v) slice — no witness needed):
5. aggregate L2  + log cost   ←  paper's choice (Assumption `positional-leakage`)
6. aggregate L1  + log cost
7. aggregate L∞  + log cost
8. aggregate L2  + raw cost

All eight methods report top-1 / top-N / mean rank, plus a union row.

In [21]:
import sys, random, itertools
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import torch.nn.functional as F
import numpy as np
from scipy.optimize import linear_sum_assignment

import config
from model import TinyTransformer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N = config.N
EPS = 1e-12
print(f'N = {N}, device = {DEVICE}')

N = 5, device = cuda


In [22]:
CHECKPOINT = PROJECT_ROOT / 'checkpoints' / 'model_2500_5.pt'

def load_model(path):
    m = TinyTransformer().to(DEVICE)
    m.load_state_dict(torch.load(path, map_location=DEVICE))
    m.eval()
    return m

model = load_model(CHECKPOINT)
print(f'Loaded {CHECKPOINT.name}')

Loaded model_2500_5.pt


/tmp/ipykernel_699622/98322929.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  m.load_state_dict(torch.load(path, map_location=DEVICE))


In [23]:
@torch.no_grad()
def sample_psi_unconstrained(model, phis):
    model.eval()
    if phis.dim() == 1:
        phis = phis.unsqueeze(0)
    B = phis.shape[0]
    psi_start = config.PSI.start
    psi_buf = torch.zeros((B, N), dtype=torch.long, device=DEVICE)
    seq = torch.cat([phis.to(DEVICE), psi_buf], dim=1)
    for j in range(N):
        logits = model(seq)
        step_logits = logits[:, psi_start - 1 + j, :]
        probs = F.softmax(step_logits, dim=-1)
        sampled = torch.multinomial(probs, num_samples=1).squeeze(-1)
        seq[:, psi_start + j] = sampled
    return seq[:, psi_start:psi_start + N]

def is_perm_rows(samples, n):
    sorted_, _ = torch.sort(samples, dim=1)
    expected = torch.arange(n, device=samples.device).unsqueeze(0).expand_as(sorted_)
    return (sorted_ == expected).all(dim=1)

@torch.no_grad()
def sample_psi_rejection(model, phis, max_attempts=50):
    model.eval()
    if phis.dim() == 1:
        phis = phis.unsqueeze(0)
    B = phis.shape[0]
    phis = phis.to(DEVICE)
    out     = torch.zeros((B, N), dtype=torch.long, device=DEVICE)
    pending = torch.ones(B,        dtype=torch.bool, device=DEVICE)
    for _ in range(max_attempts):
        idx = torch.nonzero(pending, as_tuple=False).squeeze(1)
        if idx.numel() == 0:
            break
        psis_try = sample_psi_unconstrained(model, phis[idx])
        valid = is_perm_rows(psis_try, N)
        out[idx[valid]] = psis_try[valid]
        pending[idx[valid]] = False
    if pending.any():
        raise RuntimeError(
            f'{int(pending.sum().item())} rows still invalid after {max_attempts} attempts'
        )
    return out

def marginal_matrix(samples, n=N):
    return F.one_hot(samples.long(), n).float().mean(dim=0)

def random_phis_with_constraint(B, j, u, n=N, seed=None):
    g = torch.Generator()
    if seed is not None: g.manual_seed(seed)
    other_positions = torch.tensor([i for i in range(n) if i != j], dtype=torch.long)
    other_values    = torch.tensor([v for v in range(n) if v != u], dtype=torch.long)
    keys = torch.rand((B, n - 1), generator=g)
    perms = keys.argsort(dim=1)
    perm_values = other_values[perms]
    out = torch.zeros((B, n), dtype=torch.long)
    out[:, j] = u
    out[:, other_positions] = perm_values
    return out

In [24]:
def best_assignment(C):
    C_np = C.cpu().numpy() if torch.is_tensor(C) else C
    row_ind, col_ind = linear_sum_assignment(C_np)
    return torch.tensor(col_ind, dtype=torch.long), float(C_np[row_ind, col_ind].sum())

def top_k_assignments(C, k=None):
    n = C.shape[0]
    C_np = C.cpu().numpy() if torch.is_tensor(C) else C
    rows = np.arange(n)
    scored = [
        (C_np[rows, list(perm)].sum(), perm)
        for perm in itertools.permutations(range(n))
    ]
    scored.sort(key=lambda x: x[0])
    if k is not None:
        scored = scored[:k]
    return [(torch.tensor(p, dtype=torch.long), float(c)) for c, p in scored]

## Build τ and τ_log

K1 phis per (j, u); K2 psis per phi. Per-phi marginal computed first, then both raw average and log-then-average over the K1 phis.

In [25]:
tau     = {}
tau_log = {}

seed = 42
torch.manual_seed(seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
np.random.seed(seed); random.seed(seed)

K1 = 2**7       # phis per (j, u)
K2 = 2**7         # psis per phi (large to keep log-bias small)
CHUNK = 2**14     # rows per model.forward call

for j in range(N):
    for u in range(N):
        phis = random_phis_with_constraint(K1, j=j, u=u)
        sum_p     = torch.zeros((N, N), device=DEVICE)
        sum_log_p = torch.zeros((N, N), device=DEVICE)

        if K2 <= CHUNK:
            phi_chunk = max(1, CHUNK // K2)
            psi_subchunk = K2
        else:
            phi_chunk = 1
            psi_subchunk = CHUNK

        for phi_start in range(0, K1, phi_chunk):
            P = min(phi_chunk, K1 - phi_start)
            this_phis = phis[phi_start:phi_start + P]
            counts = torch.zeros((P, N, N), device=DEVICE)
            for psi_start in range(0, K2, psi_subchunk):
                S = min(psi_subchunk, K2 - psi_start)
                phi_rep = this_phis.repeat_interleave(S, dim=0)
                psis    = sample_psi_rejection(model, phi_rep)
                psis    = psis.view(P, S, N)
                counts += F.one_hot(psis.long(), N).float().sum(dim=1)
            M_per_phi = counts / K2
            sum_p     += M_per_phi.sum(dim=0)
            sum_log_p += torch.log(M_per_phi.clamp_min(EPS)).sum(dim=0)

        avg_p     = sum_p / K1
        avg_log_p = sum_log_p / K1
        for i in range(N):
            for v in range(N):
                tau[(i, j, v, u)]     = avg_p[i, v]
                tau_log[(i, j, v, u)] = avg_log_p[i, v]
        print(f'  cell (j={j}, u={u}) done')

  cell (j=0, u=0) done
  cell (j=0, u=1) done
  cell (j=0, u=2) done
  cell (j=0, u=3) done
  cell (j=0, u=4) done
  cell (j=1, u=0) done
  cell (j=1, u=1) done
  cell (j=1, u=2) done
  cell (j=1, u=3) done
  cell (j=1, u=4) done
  cell (j=2, u=0) done
  cell (j=2, u=1) done
  cell (j=2, u=2) done
  cell (j=2, u=3) done
  cell (j=2, u=4) done
  cell (j=3, u=0) done
  cell (j=3, u=1) done
  cell (j=3, u=2) done
  cell (j=3, u=3) done
  cell (j=3, u=4) done
  cell (j=4, u=0) done
  cell (j=4, u=1) done
  cell (j=4, u=2) done
  cell (j=4, u=3) done
  cell (j=4, u=4) done


In [26]:
tau

{(0, 0, 0, 0): tensor(0.2294, device='cuda:0'),
 (0, 0, 1, 0): tensor(0.1979, device='cuda:0'),
 (0, 0, 2, 0): tensor(0.2922, device='cuda:0'),
 (0, 0, 3, 0): tensor(0.1226, device='cuda:0'),
 (0, 0, 4, 0): tensor(0.1579, device='cuda:0'),
 (1, 0, 0, 0): tensor(0.4436, device='cuda:0'),
 (1, 0, 1, 0): tensor(0.2016, device='cuda:0'),
 (1, 0, 2, 0): tensor(0.2328, device='cuda:0'),
 (1, 0, 3, 0): tensor(0.0698, device='cuda:0'),
 (1, 0, 4, 0): tensor(0.0522, device='cuda:0'),
 (2, 0, 0, 0): tensor(0.0557, device='cuda:0'),
 (2, 0, 1, 0): tensor(0.2542, device='cuda:0'),
 (2, 0, 2, 0): tensor(0.1597, device='cuda:0'),
 (2, 0, 3, 0): tensor(0.2743, device='cuda:0'),
 (2, 0, 4, 0): tensor(0.2562, device='cuda:0'),
 (3, 0, 0, 0): tensor(0.1222, device='cuda:0'),
 (3, 0, 1, 0): tensor(0.2141, device='cuda:0'),
 (3, 0, 2, 0): tensor(0.0568, device='cuda:0'),
 (3, 0, 3, 0): tensor(0.2974, device='cuda:0'),
 (3, 0, 4, 0): tensor(0.3095, device='cuda:0'),
 (4, 0, 0, 0): tensor(0.1492, device='cu

## Tensorize τ and τ_log for the aggregate methods

Build dense `(N, N, N, N)` tensors on DEVICE so the aggregate cost reduces to one broadcast subtract + reduce over `(i, v)` — no per-cell Python loops.

In [27]:
def to_tensor(tau_dict):
    """Pack {(i, j, v, u): scalar} → tensor of shape (N, N, N, N) indexed [i, j, v, u]."""
    T = torch.zeros((N, N, N, N), device=DEVICE)
    for i in range(N):
        for j in range(N):
            for v in range(N):
                for u in range(N):
                    T[i, j, v, u] = tau_dict[(i, j, v, u)]
    return T

tau_tensor     = to_tensor(tau)
tau_log_tensor = to_tensor(tau_log)
print(f'tau_tensor    : {tuple(tau_tensor.shape)}, device={tau_tensor.device}')
print(f'tau_log_tensor: {tuple(tau_log_tensor.shape)}, device={tau_log_tensor.device}')

tau_tensor    : (5, 5, 5, 5), device=cuda:0
tau_log_tensor: (5, 5, 5, 5), device=cuda:0


## Pick witnesses (for the four single-witness methods)

In [28]:
def pick_witnesses(tau_dict):
    """Return (opts_max_spread, opts_max_min_spread): each {j: (i_j, v_j)}."""
    opts_ms, opts_mm = {}, {}
    for j in range(N):
        scores_ms = torch.zeros((N, N))
        scores_mm = torch.zeros((N, N))
        for i in range(N):
            for v in range(N):
                u_func = torch.tensor([tau_dict[(i, j, v, u)].item()
                                       if torch.is_tensor(tau_dict[(i, j, v, u)])
                                       else tau_dict[(i, j, v, u)]
                                       for u in range(N)])
                scores_ms[i, v] = u_func.max() - u_func.min()
                sorted_, _ = torch.sort(u_func, descending=True)
                scores_mm[i, v] = sorted_[0] - sorted_[1]
        i_ms = (scores_ms.argmax() // N).item(); v_ms = (scores_ms.argmax() % N).item()
        i_mm = (scores_mm.argmax() // N).item(); v_mm = (scores_mm.argmax() % N).item()
        opts_ms[j], opts_mm[j] = (i_ms, v_ms), (i_mm, v_mm)
    return opts_ms, opts_mm

opts_max_spread,     opts_max_min_spread     = pick_witnesses(tau)
opts_max_spread_log, opts_max_min_spread_log = pick_witnesses(tau_log)

print('raw — opts_max_spread:    ', opts_max_spread)
print('raw — opts_max_min_spread:', opts_max_min_spread)
print('log — opts_max_spread:    ', opts_max_spread_log)
print('log — opts_max_spread:    ', opts_max_min_spread_log)

raw — opts_max_spread:     {0: (0, 2), 1: (4, 2), 2: (2, 4), 3: (2, 4), 4: (2, 1)}
raw — opts_max_min_spread: {0: (1, 0), 1: (1, 0), 2: (2, 3), 3: (2, 4), 4: (4, 0)}
log — opts_max_spread:     {0: (1, 4), 1: (2, 2), 2: (1, 1), 3: (0, 4), 4: (3, 2)}
log — opts_max_spread:     {0: (1, 0), 1: (1, 0), 2: (2, 3), 3: (0, 4), 4: (1, 0)}


## Extract over all permutations — all 8 methods

For each test phi:
1. Sample `B` psis (chunked by `B_CHUNK`); compute `mm` and `log_mm`.
2. For each of the 8 methods, build cost matrix `C[j, u]`, run brute-force ranking, record rank of truth.
3. Track top-1 / top-N / mean rank per method, plus union (any method).

**Cost matrix construction by method `kind`:**
- `raw` — single witness, raw scale: `C[j,u] = |τ[i_j,j,v_j,u] - mm[i_j,v_j]|`
- `log` — single witness, log scale: `C[j,u] = |τ_log[i_j,j,v_j,u] - log_mm[i_j,v_j]|`
- `agg_l2_log` / `agg_l1_log` / `agg_linf_log` — aggregate over (i, v) of `|log_mm - τ_log[:, j, :, u]|` under L2 / L1 / L∞
- `agg_l2_raw` — same but raw: aggregate L2 of `|mm - τ[:, j, :, u]|`

In [29]:
B = K2                                                 # psi samples per test phi
B_CHUNK = CHUNK                                           # forward-pass batch size
permutations = list(itertools.permutations(range(N)))

seed = 0
torch.manual_seed(seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
np.random.seed(seed); random.seed(seed)

# (witness_dict_or_None, tau_dict_or_None, kind)
METHODS = {
    'max-min raw':       (opts_max_min_spread,     tau,     'raw'),
    'max     raw':       (opts_max_spread,         tau,     'raw'),
    'max-min log':       (opts_max_min_spread_log, tau_log, 'log'),
    'max     log':       (opts_max_spread_log,     tau_log, 'log'),
    'aggregate L2  log': (None, None, 'agg_l2_log'),
    'aggregate L1  log': (None, None, 'agg_l1_log'),
    'aggregate L\u221E  log': (None, None, 'agg_linf_log'),
    'aggregate L2  raw': (None, None, 'agg_l2_log'),
    'aggregate L1  raw': (None, None, 'agg_l1_raw'),
    'aggregate L\u221E  raw': (None, None, 'agg_linf_raw'),
}

ranks = {m: [] for m in METHODS}
top1  = {m: 0  for m in METHODS}
topN  = {m: 0  for m in METHODS}
union_top1 = 0
union_topN = 0

def build_cost(kind, opts_dict, tau_dict, mm, log_mm):
    """Return (N, N) cost matrix on CPU."""
    if kind == 'raw':
        C = torch.zeros((N, N))
        for j in range(N):
            i_j, v_j = opts_dict[j]
            p_j = mm[i_j, v_j]
            for u in range(N):
                C[j, u] = torch.abs(tau_dict[(i_j, j, v_j, u)] - p_j)
        return C
    if kind == 'log':
        C = torch.zeros((N, N))
        for j in range(N):
            i_j, v_j = opts_dict[j]
            lp_j = log_mm[i_j, v_j]
            for u in range(N):
                C[j, u] = torch.abs(tau_dict[(i_j, j, v_j, u)] - lp_j)
        return C

    # aggregate methods: broadcast (N, N) test marginal vs. (N, N, N, N) tau tensor
    if kind == 'agg_l2_raw':
        diff = mm.unsqueeze(1).unsqueeze(3) - tau_tensor
        return (diff.abs() ** 2).sum(dim=(0, 2)).sqrt().cpu()
    if kind == 'agg_l1_raw':
        diff = mm.unsqueeze(1).unsqueeze(3) - tau_tensor
        return diff.abs().sum(dim=(0, 2)).cpu()
    if kind == 'agg_linf_raw':
        diff = mm.unsqueeze(1).unsqueeze(3) - tau_tensor
        return diff.abs().amax(dim=(0, 2)).cpu()
    if kind == 'agg_l2_log':
        diff = log_mm.unsqueeze(1).unsqueeze(3) - tau_log_tensor
        return (diff.abs() ** 2).sum(dim=(0, 2)).sqrt().cpu()
    if kind == 'agg_l1_log':
        diff = log_mm.unsqueeze(1).unsqueeze(3) - tau_log_tensor
        return diff.abs().sum(dim=(0, 2)).cpu()
    if kind == 'agg_linf_log':
        diff = log_mm.unsqueeze(1).unsqueeze(3) - tau_log_tensor
        return diff.abs().amax(dim=(0, 2)).cpu()
    raise ValueError(f'unknown kind: {kind}')

for permutation in permutations:
    base = torch.tensor(permutation, device=DEVICE).long()
    counts = torch.zeros((N, N), device=DEVICE)
    for start in range(0, B, B_CHUNK):
        S = min(B_CHUNK, B - start)
        perm_rep = base.unsqueeze(0).expand(S, -1).contiguous()
        psis = sample_psi_rejection(model, perm_rep)
        counts += F.one_hot(psis.long(), N).float().sum(dim=0)
    mm     = counts / B
    log_mm = torch.log(mm.clamp_min(EPS))

    truth_tup = tuple(permutation)
    print(f'truth = {truth_tup}')
    iter_in_top1 = iter_in_topN = False

    for name, (opts_dict, tau_dict, kind) in METHODS.items():
        C = build_cost(kind, opts_dict, tau_dict, mm, log_mm)
        full   = top_k_assignments(C, k=None)
        top_N  = full[:N]
        rank   = next(i + 1 for i, (p, _) in enumerate(full) if tuple(p.tolist()) == truth_tup)
        in_top = rank <= N
        ranks[name].append(rank)
        if rank == 1:
            top1[name] += 1
            iter_in_top1 = True
        if in_top:
            topN[name] += 1
            iter_in_topN = True
        print(f'  {name:<22}  best={full[0][0].tolist()} (cost {full[0][1]:.4f})  '
              f'rank={rank}/{len(permutations)}  in-top-{N}={in_top}')
    if iter_in_top1: union_top1 += 1
    if iter_in_topN: union_topN += 1
    print()

truth = (0, 1, 2, 3, 4)
  max-min raw             best=[2, 1, 3, 0, 4] (cost 0.5982)  rank=79/120  in-top-5=False
  max     raw             best=[3, 4, 1, 0, 2] (cost 1.1558)  rank=76/120  in-top-5=False
  max-min log             best=[2, 1, 3, 4, 0] (cost 43.4497)  rank=61/120  in-top-5=False
  max     log             best=[0, 1, 2, 4, 3] (cost 32.0967)  rank=3/120  in-top-5=True
  aggregate L2  log       best=[0, 2, 1, 3, 4] (cost 320.2176)  rank=3/120  in-top-5=True
  aggregate L1  log       best=[0, 2, 1, 3, 4] (cost 1546.4396)  rank=4/120  in-top-5=True
  aggregate L∞  log       best=[0, 2, 1, 3, 4] (cost 92.1807)  rank=5/120  in-top-5=True
  aggregate L2  raw       best=[0, 2, 1, 3, 4] (cost 320.2176)  rank=3/120  in-top-5=True
  aggregate L1  raw       best=[0, 1, 2, 3, 4] (cost 38.3085)  rank=1/120  in-top-5=True
  aggregate L∞  raw       best=[0, 1, 4, 2, 3] (cost 4.1971)  rank=3/120  in-top-5=True

truth = (0, 1, 2, 4, 3)
  max-min raw             best=[2, 3, 4, 1, 0] (cost 0

In [ ]:
T = len(permutations)
print('=' * 90)
print(f'Summary over all {T} permutations of S_{N}')
print(f'Random baselines: top-1 ~ {100/T:.1f}%, top-{N} ~ {100*N/T:.1f}%, mean rank ~ {(T+1)/2:.1f}')
print('-' * 90)
print(f'{"method":<24} {"top-1":>10} {"top-N":>10} {"mean rank":>14}')
for m in METHODS:
    print(f'{m:<24} {top1[m]}/{T:<6} {topN[m]}/{T:<6} {sum(ranks[m])/T:>12.2f}')
print('-' * 90)
print(f'{"union (any method)":<24} {union_top1}/{T:<6} {union_topN}/{T:<6}')

Summary over all 120 permutations of S_5
Random baselines: top-1 ~ 0.8%, top-5 ~ 4.2%, mean rank ~ 60.5
------------------------------------------------------------------------------------------
method                        top-1      top-N      mean rank
max-min raw              3/120    16/120           40.55
max     raw              2/120    9/120           37.51
max-min log              3/120    13/120           41.32
max     log              4/120    21/120           35.59
aggregate L2  log        27/120    59/120           14.03
aggregate L1  log        22/120    52/120           16.21
aggregate L∞  log        10/120    39/120           20.25
aggregate L2  raw        27/120    59/120           14.03
aggregate L1  raw        19/120    62/120           10.12
aggregate L∞  raw        15/120    52/120           16.78
------------------------------------------------------------------------------------------
union (any method)       51/120    99/120   


: 